# pFedMe

> The implementation of pFedMe client as in https://arxiv.org/pdf/2006.08848

In [ ]:
#| default_exp clients.pfedme

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import math
import os

from copy import deepcopy
from loguru import logger
from tqdm import tqdm

from fastcore.utils import patch
import torch
import torch.nn.functional as F

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

## pFedMe  
Personalized Federated learning using Morseu-envelope

In [ ]:
#| export
@AlgorithmRegistry.register_client("pfedme")
class pFedMeClient(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
        
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        
        self.local_model = deepcopy(self.model)    

### Training

In [ ]:
#| export
@patch
def fit(self: pFedMeClient):
    
    self.model = self.model.to(self.device)
    self.local_model = self.local_model.to(self.device)
    self.model.train()
    self.local_model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            # find an approximate theta
            for _ in range(self.cfg.K):
                self.optimizer.zero_grad()
                X, y = batch[self.data_key], batch[self.label_key]
                outputs = self.model(X)
                
                loss = self.criterion(outputs, y)
                loss.backward()
                self.persionalized_model_bar = self.optimizer.step(self.local_model)

            # update local weight after finding aproximate theta
            with torch.no_grad():
                for localweight, per_param in zip(self.local_model.parameters(), self.persionalized_model_bar):
                    localweight.sub_(self.cfg.lambda_* self.cfg.optimizer.lr * (localweight - per_param))

    with torch.no_grad():
        for model_param, local_param in zip(self.model.parameters(), self.local_model.parameters()):
            model_param.copy_(local_param)

### Evaluation

In [ ]:
#| export
@patch
def train_test_stats(self: pFedMeClient, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined

    try:
        X, y = batch[self.data_key], batch[self.label_key]
        logits = self.pers_model(X)
        loss = self.criterion(logits, y)
        probs = torch.nn.functional.softmax(logits, dim=-1)
        y_pred = probs.argmax(dim=-1)
        y_true = batch[self.label_key]

        metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true)

    except Exception as e:
        print(f"Error in _closure Eval_test_stats: {e}")
        return torch.tensor(0.0, dtype=torch.float32, device=self.device), metrics  # Return safe values

    return loss, metrics


In [ ]:
#| export
@patch
def evaluate_local(self: pFedMeClient, loader= 'train') -> dict:
    total_loss = 0
    lst_metrics = []
    pers_model = deepcopy(self.model)
    self.logger.info(f"type of model is {type(self.pers_model)}")
    pers_model.load_state_dict(self.pers_model)
    self.pers_model = pers_model.to(self.device)
    self.logger.info(f"type of model is {type(self.pers_model)}")
    self.pers_model.eval()

    num_eval = 0
    data_loader = self.train_loader if loader == 'train' else self.test_loader
    
    with torch.no_grad():
        for i, batch in enumerate(data_loader):
            batch = self.send_to_device(batch)
            loss, metrics = self.train_test_stats(batch)                 
            if not math.isnan(loss.item()):
                total_loss += loss.item()  
                num_eval += len(batch[self.data_key])  # Ensure num_eval is updated
                lst_metrics.append(metrics)           
    
    avg_loss = total_loss / num_eval if num_eval > 0 else 0.0
    self.logger.info(f"Average {loader} Loss is : {avg_loss}")
    
    if lst_metrics:
        total_metrics = {k: sum(m.get(k, 0) for m in lst_metrics) / len(lst_metrics) for k in self.cfg.test_metrics}
    else:
        total_metrics = {k: 0.0 for k in self.cfg.test_metrics}

    return {"loss": avg_loss, "metrics": total_metrics}


### Saving

In [ ]:
#| export
@patch
def save_state(self: pFedMeClient, save_to_disk= False):

    self.pers_model = deepcopy(self.model).to(self.device)
    with torch.no_grad():
        for p_model, p_updated in zip(self.pers_model.parameters(), self.persionalized_model_bar):
            p_model.copy_(p_updated)

    client_state = {
        "model": self.model.state_dict(),
        'optimizer': self.optimizer.state_dict(),
        'pers_model': self.pers_model.state_dict(),
    }

    if save_to_disk:
        state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))
        torch.save(client_state, state_path)

    return client_state
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()